# 80 — SMILES Enumeration Data Augmentation

The same molecule has many valid SMILES representations (different atom orderings).
ChemBERTa and SMILES-based models benefit from SMILES augmentation.
But it also helps RDKit-featurized LightGBM: different SMILES → different canonical
atom orderings → different hash collisions → perturbed features → ensemble effect.

Generate 10 random SMILES per training compound (10× data), train LGBM,
then at test time predict from 10 random SMILES and average.


In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and len(cp)>0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R²={r2:.4f} "
              f"r={pr:.4f} ρ={sp:.4f} τ={kt:.4f}{ca}")
    return m


In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 149


In [4]:
from rdkit import Chem

def random_smiles(smi, n=10, seed=42):
    """Generate n random SMILES representations of the same molecule."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return [smi]*n
    results = [Chem.MolToSmiles(mol)]  # canonical always first
    rng = np.random.default_rng(seed)
    for attempt in range(n * 5):
        if len(results) >= n: break
        order = list(range(mol.GetNumAtoms()))
        rng.shuffle(order)
        try:
            mol_new = Chem.RenumberAtoms(mol, order)
            rsmi = Chem.MolToSmiles(mol_new, canonical=False)
            if rsmi not in results:
                results.append(rsmi)
        except: pass
    while len(results) < n:
        results.append(results[0])
    return results[:n]

N_AUG = 8  # augmentations per training compound
print(f"Generating {N_AUG} random SMILES per training compound ({len(tr)*N_AUG:,} total)...")
aug_smiles = []
aug_y = []
for i, (smi, y_val) in enumerate(zip(tr["smiles"].tolist(), y_tr)):
    variants = random_smiles(smi, n=N_AUG, seed=i)
    aug_smiles.extend(variants)
    aug_y.extend([y_val] * len(variants))
    if (i+1) % 500 == 0: print(f"  {i+1}/{len(tr)}", flush=True)

aug_y = np.array(aug_y, dtype=np.float64)
print(f"\nAugmented training set: {len(aug_smiles):,} SMILES")


Generating 8 random SMILES per training compound (33,112 total)...


  500/4139


  1000/4139


  1500/4139


  2000/4139


  2500/4139


  3000/4139


  3500/4139


  4000/4139



Augmented training set: 33,112 SMILES

In [5]:
print("Featurizing augmented training set...", flush=True)
X_aug = impute(combined(aug_smiles))
print(f"X_aug: {X_aug.shape}")

# Standard scaffold CV on original training (not augmented) for fair comparison
print("\n=== Scaffold 5-fold CV (original vs augmented) ===", flush=True)
oof_orig = np.full(len(y_tr), np.nan)
oof_aug  = np.full(len(y_tr), np.nan)

for fold, (tr_idx, va_idx) in enumerate(splits):
    # Original
    m_o = lgb.train(LGBM, lgb.Dataset(X_tr[tr_idx], label=y_tr[tr_idx]),
                    valid_sets=[lgb.Dataset(X_tr[va_idx], label=y_tr[va_idx])],
                    callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_orig[va_idx] = m_o.predict(X_tr[va_idx])

    # Augmented (all N_AUG*len(fold_train) rows, validate on original canonical)
    aug_tr_idx = np.concatenate([np.arange(i*N_AUG, (i+1)*N_AUG) for i in tr_idx])
    aug_tr_X   = X_aug[aug_tr_idx]; aug_tr_y = aug_y[aug_tr_idx]
    m_a = lgb.train(LGBM, lgb.Dataset(aug_tr_X, label=aug_tr_y),
                    valid_sets=[lgb.Dataset(X_tr[va_idx], label=y_tr[va_idx])],
                    callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_aug[va_idx] = m_a.predict(X_tr[va_idx])

    r_o = rae(y_tr[va_idx], oof_orig[va_idx])
    r_a = rae(y_tr[va_idx], oof_aug[va_idx])
    print(f"  fold {fold+1}  orig={r_o:.4f}  augmented={r_a:.4f}", flush=True)

m_orig = full_metrics(y_tr, oof_orig, cliff_pairs, "original")
m_aug  = full_metrics(y_tr, oof_aug,  cliff_pairs, "smiles_augmented")
oof = oof_aug
print("\n" + pd.DataFrame([m_orig, m_aug], index=["original","augmented"]).round(4).to_string())


Featurizing augmented training set...


X_aug: (33112, 2265)

=== Scaffold 5-fold CV (original vs augmented) ===


  fold 1  orig=0.4982  augmented=0.4902


  fold 2  orig=0.5759  augmented=0.5638


  fold 3  orig=0.6021  augmented=0.5970


  fold 4  orig=0.5665  augmented=0.5667


  fold 5  orig=0.6033  augmented=0.5975


  [original] RAE=0.5643 MAE=0.5134 R²=0.5991 r=0.7740 ρ=0.7268 τ=0.5345  Cliff=nan
  [smiles_augmented] RAE=0.5581 MAE=0.5078 R²=0.6075 r=0.7795 ρ=0.7328 τ=0.5402  Cliff=nan

              RAE     MAE      R2  Pearson  Spearman  Kendall  Cliff_acc
original   0.5643  0.5134  0.5991   0.7740    0.7268   0.5345        NaN
augmented  0.5581  0.5078  0.6075   0.7795    0.7328   0.5402        NaN


In [6]:
# Final model + test-time augmentation
print("\nTraining final model on augmented data...", flush=True)
m_final = lgb.train(LGBM, lgb.Dataset(X_aug, label=aug_y), callbacks=[lgb.log_evaluation(-1)])

# Test-time: predict from 5 random SMILES, average
N_TEST_AUG = 5
te_preds_all = []
for aug_seed in range(N_TEST_AUG):
    te_aug_smiles = []
    for i, smi in enumerate(te["smiles"].tolist()):
        vars = random_smiles(smi, n=2, seed=1000+aug_seed*100+i)
        te_aug_smiles.append(vars[1] if len(vars)>1 else vars[0])
    X_te_aug = impute(combined(te_aug_smiles))
    te_preds_all.append(m_final.predict(X_te_aug))
te_preds = np.clip(np.mean(te_preds_all, axis=0), y_tr.min()-0.5, y_tr.max()+0.5)
print(f"Test-time augmentation: {N_TEST_AUG} SMILES variants averaged")
np.save(DATA_PROCESSED/"oof_smiles_aug.npy", oof)
np.save(DATA_PROCESSED/"te_oof_smiles_aug.npy", te_preds)
sub = pd.DataFrame({"Molecule Name":te["name"].values,"pEC50":te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"80_smiles_enumeration_aug.csv"; sub.to_csv(p,index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")



Training final model on augmented data...


Test-time augmentation: 5 SMILES variants averaged
Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\80_smiles_enumeration_aug.csv
Test: min=2.22 med=4.98 max=5.96
